# SLAM 01 — Kalman Filter & EKF: Robot Tracking from Scratch

> **Geo-Instructor · SLAM Career Track · Notebook 1 of 3**

---

## What you will build

A robot drives in a circle. Its GPS is noisy. Its odometry drifts.  
You will build **two filters** that fuse both sensors into a clean trajectory:

1. **Linear Kalman Filter** — assumes everything is Gaussian and linear  
2. **Extended Kalman Filter (EKF)** — handles the *nonlinear* robot motion model  

By the end you will see exactly why SLAM engineers reach for EKF as the baseline
before moving to factor graphs (Notebook 3).

---

## Why this matters for SLAM jobs

Every SLAM job description lists one of: EKF, UKF, particle filter, factor graph.  
They are all answers to the same question:

> **"My sensor is noisy. My motion model is wrong. Where am I?"**

The Kalman Filter is the mathematically optimal answer *when everything is Gaussian*.  
The EKF extends it to the real world via first-order linearization (the Jacobian).

---

## Prerequisites

- Python, NumPy basics
- What a mean and covariance matrix are
- Basic calculus (partial derivatives, Jacobians)

No robotics background needed. We build from first principles.

---

```
Runtime: ~5 min  |  No GPU needed  |  Works on free Colab
```

In [ ]:
# All standard — no installs needed
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms

np.random.seed(42)
plt.rcParams.update({
    'figure.facecolor': '#F4F2F0',
    'axes.facecolor':   '#F4F2F0',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.family': 'monospace',
})
print('Ready.')

---
## Part 1 — The Core Idea: Predict → Update

The Kalman Filter is a **two-step loop**:

```
        ┌─────────────────────────────────────────────────┐
        │                                                 │
        v                                                 │
  [PREDICT]  Use your motion model to guess where        │
             you will be next.  Uncertainty grows.       │
        │                                                 │
        v                                                 │
  [UPDATE]   A noisy sensor arrives.  Fuse it with       │
             the prediction.  Uncertainty shrinks.       │
        │                                                 │
        └─────────────────── repeat ────────────────────-┘
```

Your **state** is not a single point — it is a **Gaussian**: a mean `x` and a covariance `P`.

```
  Belief at time t:  N( x_t,  P_t )
                       mean   uncertainty
```

In [ ]:
# ── Visualize what a 2-D Gaussian belief looks like ──────────────────────────

def draw_covariance_ellipse(ax, mean, cov, n_std=2.0, color='steelblue', alpha=0.3, label=''):
    """Draw the n_std-sigma confidence ellipse for a 2x2 covariance matrix."""
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = eigvals.argsort()[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * n_std * np.sqrt(eigvals)
    ell = Ellipse(xy=mean, width=width, height=height, angle=angle,
                  facecolor=color, alpha=alpha, edgecolor=color, linewidth=1.5, label=label)
    ax.add_patch(ell)
    ax.plot(*mean, 'o', color=color, markersize=6)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.suptitle('The robot\'s belief is a Gaussian — mean + covariance', fontsize=11)

scenarios = [
    ('Large uncertainty\n(just started)', [[4,0],[0,4]], 'tomato'),
    ('Medium uncertainty\n(some sensor data)', [[1.5,0.4],[0.4,0.8]], 'steelblue'),
    ('Small uncertainty\n(many updates fused)', [[0.3,0],[0,0.2]], 'seagreen'),
]
mean = [0, 0]
for ax, (title, cov, color) in zip(axes, scenarios):
    draw_covariance_ellipse(ax, mean, np.array(cov), n_std=2, color=color)
    ax.set_xlim(-6, 6); ax.set_ylim(-6, 6)
    ax.set_aspect('equal'); ax.set_title(title, fontsize=9)
    ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
    ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)

plt.tight_layout()
plt.show()
print('The 2-sigma ellipse contains ~95% of the probability mass.')

---
## Part 2 — Linear Kalman Filter: 1-D Robot on a Rail

Before going to 2-D nonlinear motion, let's build intuition on the simplest case:
a robot moving in 1-D. State is position `x` and velocity `v`.

### The 5 equations

**PREDICT (propagate forward in time)**
```
  x_pred = F * x          # apply motion model
  P_pred = F * P * F.T + Q   # uncertainty grows (add process noise Q)
```

**UPDATE (fuse sensor measurement z)**
```
  y = z - H * x_pred      # innovation: what the sensor *actually* said vs prediction
  S = H * P_pred * H.T + R  # innovation covariance (sensor noise R)
  K = P_pred * H.T * S^-1  # Kalman gain: how much to trust the sensor
  x = x_pred + K * y      # update state
  P = (I - K * H) * P_pred  # update uncertainty (it shrinks!)
```

The **Kalman gain K** is the soul of the filter:
- K near 1 → trust the sensor completely (low R, high P_pred)
- K near 0 → trust the prediction (high R, low P_pred)

In [ ]:
# ── Simulate a 1-D robot: constant velocity, noisy GPS ───────────────────────
dt = 0.1        # time step (seconds)
T  = 200        # number of steps
true_vel = 1.0  # m/s

# State: [position, velocity]
# Motion model: x_{t+1} = F * x_t  (constant velocity)
F = np.array([[1, dt],
              [0,  1]])

# Observation model: we only observe position (not velocity)
H = np.array([[1, 0]])

# Noise covariances
Q = np.array([[0.01*dt**2, 0],   # process noise (model imperfection)
              [0,          0.01]])
R = np.array([[5.0]])             # GPS measurement noise (2-sigma ~4.5m)

# Generate ground truth and noisy GPS
true_positions = np.arange(T) * dt * true_vel
gps_noise = np.random.randn(T) * np.sqrt(R[0, 0])
gps_measurements = true_positions + gps_noise

# ── Kalman Filter ─────────────────────────────────────────────────────────────
x = np.array([0.0, 0.0])         # initial state estimate
P = np.eye(2) * 500.0            # initial (very uncertain) covariance
I = np.eye(2)

estimates = []
variances  = []
gains      = []

for z_raw in gps_measurements:
    z = np.array([z_raw])

    # PREDICT
    x = F @ x
    P = F @ P @ F.T + Q

    # UPDATE
    y = z - H @ x
    S = H @ P @ H.T + R
    K = P @ H.T @ np.linalg.inv(S)
    x = x + K @ y
    P = (I - K @ H) @ P

    estimates.append(x[0])          # store estimated position
    variances.append(P[0, 0])       # store position variance
    gains.append(K[0, 0])           # store Kalman gain

estimates = np.array(estimates)
variances = np.array(variances)

# ── Plot ──────────────────────────────────────────────────────────────────────
t = np.arange(T) * dt
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 10))
fig.suptitle('Linear Kalman Filter: 1-D Robot Tracking', fontsize=12, y=0.98)

ax1.plot(t, true_positions, 'k-', lw=2, label='Ground truth')
ax1.scatter(t, gps_measurements, s=4, c='tomato', alpha=0.5, label=f'GPS (noise sigma={np.sqrt(R[0,0]):.1f}m)')
ax1.plot(t, estimates, 'steelblue', lw=2, label='KF estimate')
ax1.fill_between(t,
                 estimates - 2*np.sqrt(variances),
                 estimates + 2*np.sqrt(variances),
                 alpha=0.2, color='steelblue', label='2-sigma band')
ax1.set_ylabel('Position (m)'); ax1.legend(fontsize=8)

rmse_kf  = np.sqrt(np.mean((estimates - true_positions)**2))
rmse_gps = np.sqrt(np.mean((gps_measurements - true_positions)**2))
ax1.set_title(f'KF RMSE: {rmse_kf:.3f}m   |   Raw GPS RMSE: {rmse_gps:.3f}m', fontsize=9)

ax2.plot(t, variances, 'darkorange', lw=1.5)
ax2.set_ylabel('Position variance P[0,0]')
ax2.set_title('Uncertainty converges quickly — filter becomes confident', fontsize=9)

ax3.plot(t, gains, 'purple', lw=1.5)
ax3.set_ylabel('Kalman gain K')
ax3.set_xlabel('Time (s)')
ax3.set_title('Kalman gain: starts high (trust sensor), settles (balance prediction & sensor)', fontsize=9)

plt.tight_layout()
plt.show()
print(f'KF reduced GPS error by {(1 - rmse_kf/rmse_gps)*100:.1f}%')

---
## Part 3 — The Problem with Linear KF: Nonlinear Motion

Real robots move in **curves**. A robot turning with constant angular velocity has state:

```
  x = [px, py, theta, v, omega]  (position, heading, speed, turn rate)
```

The motion update is:
```
  px'    = px + (v/omega) * (sin(theta + omega*dt) - sin(theta))
  py'    = py - (v/omega) * (cos(theta + omega*dt) - cos(theta))
  theta' = theta + omega * dt
  v'     = v
  omega' = omega
```

This is **nonlinear** — `sin` and `cos` of the state. The linear KF breaks down.

### The EKF fix: linearize locally with the Jacobian

Instead of using `F` (a constant matrix), the EKF computes the **Jacobian** of
the motion function at the current state estimate — a linear approximation that
is re-evaluated every step.

```
  F_jac = df/dx  evaluated at current x_hat
```

In [ ]:
# ── EKF for a 2-D robot driving in a circle ───────────────────────────────────
# State: [x, y, theta, v, omega]  (5-D)
# Sensors: GPS (x,y) + compass (theta)  — 3-D observation

def motion_model(state, dt):
    """Nonlinear constant-turn-rate-and-velocity (CTRV) motion model."""
    x, y, theta, v, omega = state
    if abs(omega) < 1e-6:  # straight-line motion (avoid divide by zero)
        return np.array([
            x + v * np.cos(theta) * dt,
            y + v * np.sin(theta) * dt,
            theta,
            v,
            omega,
        ])
    return np.array([
        x + (v/omega) * (np.sin(theta + omega*dt) - np.sin(theta)),
        y - (v/omega) * (np.cos(theta + omega*dt) - np.cos(theta)),
        theta + omega * dt,
        v,
        omega,
    ])

def motion_jacobian(state, dt):
    """Jacobian of motion_model w.r.t. state — the 'F' in the EKF."""
    x, y, theta, v, omega = state
    J = np.eye(5)
    if abs(omega) < 1e-6:
        J[0, 2] = -v * np.sin(theta) * dt
        J[0, 3] =  np.cos(theta) * dt
        J[1, 2] =  v * np.cos(theta) * dt
        J[1, 3] =  np.sin(theta) * dt
    else:
        J[0, 2] = (v/omega) * (np.cos(theta + omega*dt) - np.cos(theta))
        J[0, 3] = (1/omega) * (np.sin(theta + omega*dt) - np.sin(theta))
        J[0, 4] = (v/omega**2) * (np.sin(theta) - np.sin(theta + omega*dt)) \
                  + (v/omega) * np.cos(theta + omega*dt) * dt
        J[1, 2] = (v/omega) * (np.sin(theta + omega*dt) - np.sin(theta))
        J[1, 3] = (1/omega) * (-np.cos(theta + omega*dt) + np.cos(theta))
        J[1, 4] = (v/omega**2) * (np.cos(theta + omega*dt) - np.cos(theta)) \
                  + (v/omega) * np.sin(theta + omega*dt) * dt
        J[2, 4] = dt
    return J

# Observation model: we measure [x, y, theta] with GPS + compass
H_ekf = np.array([[1,0,0,0,0],
                  [0,1,0,0,0],
                  [0,0,1,0,0]])

print('Motion model and Jacobian defined.')
print('State dimension: 5  [x, y, theta, v, omega]')
print('Observation dimension: 3  [x, y, theta]')

In [ ]:
# ── Simulate ground truth ─────────────────────────────────────────────────────
dt   = 0.1
T    = 300
R_radius = 5.0   # circle radius in meters
v_true   = 1.0   # m/s
om_true  = v_true / R_radius  # omega = v/r => circle

true_states = np.zeros((T, 5))
true_states[0] = [R_radius, 0, np.pi/2, v_true, om_true]
for i in range(1, T):
    true_states[i] = motion_model(true_states[i-1], dt)

# ── Generate noisy sensor readings ───────────────────────────────────────────
gps_sigma     = 0.5   # meters
compass_sigma = 0.05  # radians (~3 deg)

measurements = np.zeros((T, 3))
measurements[:, 0] = true_states[:, 0] + np.random.randn(T) * gps_sigma
measurements[:, 1] = true_states[:, 1] + np.random.randn(T) * gps_sigma
measurements[:, 2] = true_states[:, 2] + np.random.randn(T) * compass_sigma

# ── EKF ───────────────────────────────────────────────────────────────────────
# Process noise: mainly uncertainty in v and omega commands
Q_ekf = np.diag([0.05**2, 0.05**2, 0.01**2, 0.1**2, 0.01**2])
R_ekf = np.diag([gps_sigma**2, gps_sigma**2, compass_sigma**2])

# Initial estimate: noisy start
x_est = true_states[0].copy()
x_est[:2] += np.random.randn(2) * 1.0   # wrong initial position
P_est = np.diag([2.0, 2.0, 0.5, 1.0, 0.1])

I5 = np.eye(5)
ekf_states   = np.zeros((T, 5))
ekf_states[0] = x_est

for i in range(1, T):
    # PREDICT
    x_pred = motion_model(x_est, dt)
    F_jac  = motion_jacobian(x_est, dt)
    P_pred = F_jac @ P_est @ F_jac.T + Q_ekf

    # UPDATE
    z  = measurements[i]
    y  = z - H_ekf @ x_pred
    # Wrap angle innovation to [-pi, pi]
    y[2] = (y[2] + np.pi) % (2*np.pi) - np.pi

    S  = H_ekf @ P_pred @ H_ekf.T + R_ekf
    K  = P_pred @ H_ekf.T @ np.linalg.inv(S)
    x_est = x_pred + K @ y
    P_est = (I5 - K @ H_ekf) @ P_pred

    ekf_states[i] = x_est

print('EKF done.')
pos_err_ekf = np.sqrt(np.sum((ekf_states[:,:2] - true_states[:,:2])**2, axis=1))
pos_err_gps = np.sqrt(np.sum((measurements[:,:2] - true_states[:,:2])**2, axis=1))
print(f'Mean position error  — EKF:       {pos_err_ekf.mean():.3f} m')
print(f'Mean position error  — Raw GPS:   {pos_err_gps.mean():.3f} m')

In [ ]:
# ── Visualize ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Extended Kalman Filter: 2-D Robot Tracking a Circle', fontsize=12)

ax = axes[0]
ax.plot(true_states[:,0], true_states[:,1], 'k-', lw=2, label='Ground truth', zorder=5)
ax.scatter(measurements[:,0], measurements[:,1], s=4, c='tomato', alpha=0.4, label='GPS (noisy)')
ax.plot(ekf_states[:,0], ekf_states[:,1], 'steelblue', lw=2, label='EKF estimate')

# Draw a few covariance ellipses along the path
# (re-run EKF briefly to collect covariances — just store P at intervals)
# For simplicity, show the final one conceptually:
ax.plot(ekf_states[0,0], ekf_states[0,1], 'gs', ms=10, label='Start (wrong init)', zorder=10)
ax.plot(ekf_states[-1,0], ekf_states[-1,1], 'g^', ms=10, label='End', zorder=10)

ax.set_aspect('equal'); ax.legend(fontsize=8)
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title('Trajectory')

ax2 = axes[1]
t = np.arange(T) * dt
ax2.plot(t, pos_err_ekf, 'steelblue', lw=1.5, label='EKF error')
ax2.plot(t, pos_err_gps, 'tomato', lw=1, alpha=0.6, label='Raw GPS error')
ax2.axhline(pos_err_ekf.mean(), color='steelblue', ls='--', lw=1, label=f'EKF mean={pos_err_ekf.mean():.3f}m')
ax2.axhline(pos_err_gps.mean(), color='tomato', ls='--', lw=1, label=f'GPS mean={pos_err_gps.mean():.3f}m')
ax2.set_xlabel('Time (s)'); ax2.set_ylabel('Position error (m)')
ax2.set_title('EKF vs raw GPS error over time')
ax2.legend(fontsize=8)
ax2.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

---
## Part 4 — Checking the Jacobian with Finite Differences

A common EKF bug is a wrong Jacobian. The correct way to verify:
**compare the analytic Jacobian to the numerical one computed by finite differences.**

In [ ]:
def numerical_jacobian(f, x, dt, eps=1e-5):
    """Compute Jacobian of f(x) numerically via central differences."""
    n = len(x)
    J = np.zeros((n, n))
    for i in range(n):
        x_plus  = x.copy(); x_plus[i]  += eps
        x_minus = x.copy(); x_minus[i] -= eps
        J[:, i] = (f(x_plus, dt) - f(x_minus, dt)) / (2 * eps)
    return J

# Check at a random state
test_state = np.array([3.0, 1.5, 0.8, 1.2, 0.3])
dt_test    = 0.1

J_analytic  = motion_jacobian(test_state, dt_test)
J_numerical = numerical_jacobian(motion_model, test_state, dt_test)

max_diff = np.max(np.abs(J_analytic - J_numerical))
print(f'Max |analytic - numerical| Jacobian element: {max_diff:.2e}')
print('(Should be < 1e-5 for a correct Jacobian)\n')

print('Analytic Jacobian:')
print(np.array2string(J_analytic, precision=4, suppress_small=True))
print('\nNumerical Jacobian:')
print(np.array2string(J_numerical, precision=4, suppress_small=True))

---
## Part 5 — Exercises (SLAM interview prep)

### Exercise 1 — Change the sensor
Replace the GPS+compass with a **range-bearing sensor** that measures the distance
and angle to a known landmark at position `[10, 0]`.

```python
# Hint: observation model becomes:
# z = [sqrt((x-lx)**2 + (y-ly)**2),  atan2(y-ly, x-lx) - theta]
# You need a new H_jac (Jacobian of this observation function).
```

### Exercise 2 — Tune the noise matrices
Try setting `Q_ekf` very small (you trust the motion model perfectly).  
What happens? Now set `R_ekf` very small (you trust the sensor perfectly).  
Plot both cases. What is the filter trading off?

### Exercise 3 — Simulate GPS outage
At `t=10s` to `t=15s`, stop providing GPS updates (only use prediction).  
How much does the position error grow? How fast does the covariance ellipse expand?

### Exercise 4 — Compare to UKF
The Unscented Kalman Filter (UKF) avoids computing Jacobians by using **sigma points**.  
Look up the UKF algorithm and implement it for the same 2-D tracking problem.  
Compare accuracy to EKF on tight turns (high omega).

---
## Summary

| Concept | What it does | SLAM relevance |
|---------|-------------|----------------|
| Kalman gain K | Weights sensor vs prediction | Sensor fusion architecture |
| Covariance P | Tracks uncertainty | Localization confidence |
| Jacobian F_jac | Linearizes nonlinear motion | EKF correctness |
| Innovation y | Residual after prediction | Loop closure signal |
| Process noise Q | Uncertainty in motion model | Odometry calibration |
| Measurement noise R | Sensor accuracy | LiDAR / IMU calibration |

**Next:** Notebook 2 — ICP: Aligning Two Point Clouds